# Identificación y autoría

**Speaker:** Eduardo Osorio Venegas

**Apoyo de inteligencia artificial generativa:** ChatGPT (OpenAI), para la elaboración de las celdas Markdown.

Material educativo y demostrativo, sujeto a revisión y validación en Microsoft Fabric.

# Configurar Modelo

In [1]:
import synapse.ml.spark.aifunc as aifunc

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    LongType,
    DoubleType,
    IntegerType
)

from datetime import datetime, timezone
import uuid
import json

# Configurar el modelo
MODEL_NAME = "gpt-5-mini"

aifunc.default_conf.set_deployment_name(MODEL_NAME)

print(f"Modelo configurado: {MODEL_NAME}")


StatementMeta(, 884ed155-8743-4b49-91f5-d03114af4d0c, 3, Finished, Available, Finished, False)

Modelo configurado: gpt-5-mini


# Registros ficticios

In [2]:

data = [
    (
        1,
        "Microsoft Fabric es una plataforma SaaS de analítica "
        "que integra ingeniería de datos, ciencia de datos y BI."
    ),
    (
        2,
        "OneLake permite centralizar los datos de una organización "
        "en una estructura de almacenamiento unificada."
    ),
    (
        3,
        "Delta Lake permite almacenar datos mediante transacciones "
        "ACID y control de versiones."
    ),
    (
        4,
        "Microsoft Purview permite implementar capacidades "
        "de gobierno, catálogo y clasificación de datos."
    ),
    (
        5,
        "PySpark permite procesar grandes volúmenes de datos "
        "mediante procesamiento distribuido."
    )
]

df = spark.createDataFrame(
    data,
    ["id", "descripcion"]
)

display(df)


StatementMeta(, 884ed155-8743-4b49-91f5-d03114af4d0c, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 21774af0-2471-49fe-9551-f8713ec702f3)

# Ejecutar AI Functions

* Explicación con ia generativa

In [3]:

run_id = str(uuid.uuid4())

# Ejecutar la función de IA
resultado = df.ai.generate_response(
    prompt=(
        "Explica el siguiente concepto tecnológico "
        "en un máximo de 40 palabras: {descripcion}"
    ),
    is_prompt_template=True,
    output_col="respuesta"
)

# Mostrar los resultados
display(resultado)


StatementMeta(, 884ed155-8743-4b49-91f5-d03114af4d0c, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, a33a7c38-8b42-4dd3-ac97-e71e0f023537)

# Validar el consumo de tokens

###### **ai.stats** url: https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/pyspark/configuration?tabs=llm-global-pyspark%2Cembedding-global-pyspark%2Cfoundry-openai-pyspark

###### **Diccionario de campos**

* num_successful:	Filas procesadas correctamente
* num_exceptions:	Filas con excepciones
* num_unevaluated:	Filas no evaluadas
* num_harmful:	Filas bloqueadas por filtros
* input_tokens:	Tokens de entrada
* cached_tokens:	Tokens de entrada en caché
* output_tokens:	Tokens generados
* reasoning_tokens:	Tokens de razonamiento
* model: Modelo utilizado

In [4]:

# Obtener estadísticas de la ejecución
stats_df = resultado.ai.stats

# Mostrar estadísticas originales
display(stats_df)

# Recuperar las métricas
stats_rows = stats_df.collect()

if not stats_rows:
    raise RuntimeError(
        "AI Functions no devolvió estadísticas de ejecución."
    )

# Normalmente existe un registro agregado por operación
stats = stats_rows[0].asDict(recursive=True)

print("===== VALIDACIÓN AI FUNCTIONS =====")

for key, value in stats.items():
    print(f"{key}: {value}")

StatementMeta(, 884ed155-8743-4b49-91f5-d03114af4d0c, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, cdbe4dac-b1a8-4765-a833-0b130bad616c)

===== VALIDACIÓN AI FUNCTIONS =====
num_successful: 5
num_exceptions: 0
num_unevaluated: 0
cached_tokens: 0
input_tokens: 241
output_tokens: 1447
reasoning_tokens: 1024
client_type: fabric_llm_endpoint
model: gpt-5-mini


# Cálculo aproximado de capacidad

**Referencia de facturación:** https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/billing?tabs=pandas

Microsoft Fabric AI Functions permite estimar el consumo de capacidad asociado a las solicitudes realizadas a los modelos de inteligencia artificial. Este consumo se expresa en **CU-segundos (Capacity Units por segundo)** y depende de la cantidad y el tipo de tokens procesados durante cada ejecución.

Para realizar esta estimación se utiliza la siguiente fórmula, considerando las tarifas de `gpt-5-mini`:

$$
CU_s = \frac{(I-C)\times 8.40 + C\times 0.84 + O\times 67.23}{1000}
$$

**Variables de la fórmula:**

- **I (Input Tokens):** cantidad total de tokens de entrada enviados al modelo.
- **C (Cached Input Tokens):** cantidad de tokens de entrada reutilizados desde la caché.
- **O (Output Tokens):** cantidad de tokens generados como respuesta por el modelo.

**Descripción del cálculo:**

El cálculo separa los tokens de entrada en dos categorías: tokens procesados normalmente y tokens recuperados desde la caché. Esto permite aplicar una tarifa diferenciada a cada categoría.

1. **Entrada normal:** se restan los tokens en caché del total de tokens de entrada `(I - C)` y se multiplican por la tarifa de 8,40 CU-segundos por cada 1.000 tokens.
2. **Entrada en caché:** los tokens reutilizados desde la caché `(C)` se multiplican por la tarifa reducida de 0,84 CU-segundos por cada 1.000 tokens.
3. **Salida del modelo:** los tokens generados por el modelo `(O)` se multiplican por la tarifa de 67,23 CU-segundos por cada 1.000 tokens.
4. **Consumo total:** se suman los tres componentes y se divide el resultado por 1.000 para obtener el consumo estimado expresado en CU-segundos.

**Consideraciones:**

- La estimación utiliza las tarifas correspondientes al modelo `gpt-5-mini`.
- El consumo puede variar según el modelo utilizado y las tarifas de facturación vigentes.
- Los tokens de entrada en caché deben estar incluidos dentro del total de tokens de entrada, no contabilizarse como una categoría adicional.
- El resultado representa una estimación del consumo asociado al procesamiento de tokens, no necesariamente el consumo total de capacidad de Microsoft Fabric de todos los componentes de la ejecución.


In [5]:
# Extraer métricas de forma segura
input_tokens = int(stats.get("input_tokens") or 0)
cached_tokens = int(stats.get("cached_tokens") or 0)
output_tokens = int(stats.get("output_tokens") or 0)
reasoning_tokens = int(stats.get("reasoning_tokens") or 0)

# Evitar contar más tokens en caché que tokens de entrada
cached_tokens = min(cached_tokens, input_tokens)

regular_input_tokens = input_tokens - cached_tokens

# Tarifas documentadas para gpt-5-mini
INPUT_RATE = 8.40
CACHED_RATE = 0.84
OUTPUT_RATE = 67.23

estimated_cu_seconds = (
    regular_input_tokens * INPUT_RATE
    + cached_tokens * CACHED_RATE
    + output_tokens * OUTPUT_RATE
) / 1000

print("===== CONSUMO DEL MODELO =====")
print(f"Modelo: {stats.get('model', MODEL_NAME)}")
print(f"Tokens de entrada: {input_tokens:,}")
print(f"Tokens en caché: {cached_tokens:,}")
print(f"Tokens de salida: {output_tokens:,}")
print(f"Tokens de razonamiento: {reasoning_tokens:,}")
print(f"CU-segundos estimados: {estimated_cu_seconds:.4f}")

StatementMeta(, 884ed155-8743-4b49-91f5-d03114af4d0c, 7, Finished, Available, Finished, False)

===== CONSUMO DEL MODELO =====
Modelo: gpt-5-mini
Tokens de entrada: 241
Tokens en caché: 0
Tokens de salida: 1,447
Tokens de razonamiento: 1,024
CU-segundos estimados: 99.3062


# Crear tabla de auditoria

* run_id:	Identificador único de ejecución
* execution_timestamp:	Fecha y hora UTC
* notebook_name:	Notebook ejecutado
* operation:	Función AI utilizada
* model_configured:	Modelo configurado
* model_reported:	Modelo informado por Fabric
* num_input_rows:	Filas enviadas
* num_successful:	Procesamientos exitosos
* num_exceptions:	Excepciones
* num_unevaluated:	Filas no evaluadas
* num_harmful:	Filas bloqueadas
* input_tokens:	Tokens de entrada
* cached_tokens:	Tokens en caché
* output_tokens:	Tokens de salida
* reasoning_tokens:	Tokens de razonamiento
* estimated_cu_seconds:	CU-segundos estimados
* client_type:	Tipo de endpoint
* execution_status:	Estado de la ejecución

In [7]:

# Obtener estadísticas de ejecución
num_input_rows = df.count()

def metric(name):
    value = stats.get(name)
    return int(value) if value is not None else 0

successful = metric("num_successful")
exceptions = metric("num_exceptions")
unevaluated = metric("num_unevaluated")
harmful = metric("num_harmful")

model_reported = str(
    stats.get("model") or MODEL_NAME
)

client_type = str(
    stats.get("client_type") or "unknown"
)

# Estado de la ejecución
if exceptions > 0 or harmful > 0:
    execution_status = "PARTIAL"
elif successful == num_input_rows:
    execution_status = "SUCCESS"
else:
    execution_status = "INCOMPLETE"

# Esquema de auditoría
audit_schema = StructType([
    StructField("run_id", StringType(), False),
    StructField("execution_timestamp", StringType(), False),
    StructField("notebook_name", StringType(), True),
    StructField("operation", StringType(), True),
    StructField("model_configured", StringType(), True),
    StructField("model_reported", StringType(), True),
    StructField("num_input_rows", LongType(), True),
    StructField("num_successful", LongType(), True),
    StructField("num_exceptions", LongType(), True),
    StructField("num_unevaluated", LongType(), True),
    StructField("num_harmful", LongType(), True),
    StructField("input_tokens", LongType(), True),
    StructField("cached_tokens", LongType(), True),
    StructField("output_tokens", LongType(), True),
    StructField("reasoning_tokens", LongType(), True),
    StructField("estimated_cu_seconds", DoubleType(), True),
    StructField("client_type", StringType(), True),
    StructField("execution_status", StringType(), True)
])

audit_record = [(
    run_id,
    datetime.now(timezone.utc).isoformat(),
    "NB_AIFUCTIONS_CONTROL",
    "generate_response",
    MODEL_NAME,
    model_reported,
    num_input_rows,
    successful,
    exceptions,
    unevaluated,
    harmful,
    input_tokens,
    cached_tokens,
    output_tokens,
    reasoning_tokens,
    float(estimated_cu_seconds),
    client_type,
    execution_status
)]

audit_df = spark.createDataFrame(
    audit_record,
    schema=audit_schema
)

display(audit_df)


StatementMeta(, 884ed155-8743-4b49-91f5-d03114af4d0c, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, b7e76298-72d0-418d-ad89-1a565c2e104a)

# Persistir tabla de auditoria

In [ ]:

TABLE_NAME = "dbo.audit_ai_functions"

(
    audit_df.write
    .format("delta")
    .mode("append")
    .saveAsTable(TABLE_NAME)
)

print(f"Auditoría guardada correctamente: {TABLE_NAME}")
